In [7]:
!pip install pandas lxml

In [3]:
import random
import math
import re
import pandas as pd
from copy import deepcopy
from collections import defaultdict

# ============================================================
# CONFIG
# ============================================================

NUM_SIMULATIONS = 10000
PRINT_SINGLE_TOURNAMENT = True
RANDOM_SEED = None

if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)

HOST_BONUS = {
    "USA": 20,
    "Mexico": 20,
    "Canada": 20,
}

BASE_K_GROUP = 20
BASE_K_KNOCKOUT = 25
BASE_K_FINAL = 30

THIRD_PLACE_TABLE_URL = "https://en.wikipedia.org/wiki/Template:2026_FIFA_World_Cup_third-place_table"
THIRD_PLACE_SLOTS = ["1A", "1B", "1D", "1E", "1G", "1I", "1K", "1L"]

# ============================================================
# TEAMS
# ============================================================

groups = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["USA", "Paraguay", "Australia", "Turkiye"],
    "E": ["Germany", "Curacao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

base_team_elo = {
    "Brazil": 1984, "France": 2081, "Argentina": 2113, "Germany": 1923,
    "England": 2020, "Spain": 2165, "Portugal": 1984,
    "Bosnia and Herzegovina": 1594,
    "Belgium": 1867, "Netherlands": 1961, "Croatia": 1930, "USA": 1721,
    "Japan": 1904, "Senegal": 1878, "Uruguay": 1892, "Morocco": 1822,
    "Canada": 1784, "Switzerland": 1889, "Czech Republic": 1726,
    "South Korea": 1752, "Mexico": 1860, "Iran": 1760,
    "Colombia": 1975, "Ecuador": 1933,
    "Australia": 1783, "Ghana": 1503, "Qatar": 1425,
    "South Africa": 1524, "Paraguay": 1833, "Panama": 1737,
    "Iraq": 1607, "Jordan": 1690, "Algeria": 1743,
    "Tunisia": 1636, "New Zealand": 1585, "Saudi Arabia": 1568,
    "Ivory Coast": 1676, "Uzbekistan": 1727, "Austria": 1827,
    "Egypt": 1689, "Norway": 1912, "Curacao": 1436,
    "DR Congo": 1655, "Haiti": 1532, "Turkiye": 1902,
    "Sweden": 1719, "Cape Verde": 1549, "Scotland": 1767,
}
# ============================================================
# VENUES AND TIMES
# ============================================================

MATCH_SCHEDULE = {

    73: {
        "date": "Sunday 2026-06-28",
        "uk_time": "20:00",
        "venue": "SoFi Stadium",
        "city": "Inglewood",
        "stage": "Round of 32",
        "slot1": "2A",
        "slot2": "2B",
    },

    74: {
        "date": "Monday 2026-06-29",
        "uk_time": "21:30",
        "venue": "Gillette Stadium",
        "city": "Foxborough",
        "stage": "Round of 32",
        "slot1": "1E",
        "slot2": "3A, 3B, 3C, 3D, 3F",
    },

    75: {
        "date": "Tuesday 2026-06-30",
        "uk_time": "02:00",
        "venue": "Estadio BBVA",
        "city": "Guadalupe",
        "stage": "Round of 32",
        "slot1": "1F",
        "slot2": "2C",
    },

    76: {
        "date": "Monday 2026-06-29",
        "uk_time": "18:00",
        "venue": "NRG Stadium",
        "city": "Houston",
        "stage": "Round of 32",
        "slot1": "1C",
        "slot2": "2F",
    },

    77: {
        "date": "Tuesday 2026-06-30",
        "uk_time": "22:00",
        "venue": "MetLife Stadium",
        "city": "East Rutherford",
        "stage": "Round of 32",
        "slot1": "1I",
        "slot2": "3G, 3H, 3C, 3D, 3F",
    },

    78: {
        "date": "Tuesday 2026-06-30",
        "uk_time": "18:00",
        "venue": "AT&T Stadium",
        "city": "Arlington",
        "stage": "Round of 32",
        "slot1": "2E",
        "slot2": "2I",
    },

    79: {
        "date": "Wednesday 2026-07-01",
        "uk_time": "02:00",
        "venue": "Estadio Azteca",
        "city": "Mexico City",
        "stage": "Round of 32",
        "slot1": "1A",
        "slot2": "3E, 3H, 3C, 3I, 3F",
    },

    80: {
        "date": "Wednesday 2026-07-01",
        "uk_time": "17:00",
        "venue": "Mercedes-Benz Stadium",
        "city": "Atlanta",
        "stage": "Round of 32",
        "slot1": "1L",
        "slot2": "3E, 3H, 3I, 3J, 3K",
    },

    81: {
        "date": "Thursday 2026-07-02",
        "uk_time": "01:00",
        "venue": "Levi's Stadium",
        "city": "Santa Clara",
        "stage": "Round of 32",
        "slot1": "1D",
        "slot2": "3E, 3B, 3F, 3J, 3I",
    },

    82: {
        "date": "Wednesday 2026-07-01",
        "uk_time": "21:00",
        "venue": "Lumen Field",
        "city": "Seattle",
        "stage": "Round of 32",
        "slot1": "1G",
        "slot2": "3E, 3H, 3I, 3J, 3A",
    },

    83: {
        "date": "Friday 2026-07-03",
        "uk_time": "00:00",
        "venue": "BMO Field",
        "city": "Toronto",
        "stage": "Round of 32",
        "slot1": "2K",
        "slot2": "2L",
    },

    84: {
        "date": "Thursday 2026-07-02",
        "uk_time": "20:00",
        "venue": "SoFi Stadium",
        "city": "Inglewood",
        "stage": "Round of 32",
        "slot1": "1H",
        "slot2": "2J",
    },

    85: {
        "date": "Friday 2026-07-03",
        "uk_time": "04:00",
        "venue": "BC Place",
        "city": "Vancouver",
        "stage": "Round of 32",
        "slot1": "1B",
        "slot2": "3E, 3F, 3G, 3J, 3I",
    },

    86: {
        "date": "Friday 2026-07-03",
        "uk_time": "23:00",
        "venue": "Hard Rock Stadium",
        "city": "Miami Gardens",
        "stage": "Round of 32",
        "slot1": "1J",
        "slot2": "2H",
    },

    87: {
        "date": "Saturday 2026-07-04",
        "uk_time": "02:30",
        "venue": "Arrowhead Stadium",
        "city": "Kansas City",
        "stage": "Round of 32",
        "slot1": "1K",
        "slot2": "3E, 3D, 3I, 3J, 3L",
    },

    88: {
        "date": "Friday 2026-07-03",
        "uk_time": "19:00",
        "venue": "AT&T Stadium",
        "city": "Arlington",
        "stage": "Round of 32",
        "slot1": "2D",
        "slot2": "2G",
    },

    89: {
        "date": "Saturday 2026-07-04",
        "uk_time": "22:00",
        "venue": "Lincoln Financial Field",
        "city": "Philadelphia",
        "stage": "Round of 16",
        "slot1": "Winner 74",
        "slot2": "Winner 77",
    },

    90: {
        "date": "Saturday 2026-07-04",
        "uk_time": "18:00",
        "venue": "NRG Stadium",
        "city": "Houston",
        "stage": "Round of 16",
        "slot1": "Winner 73",
        "slot2": "Winner 75",
    },

    91: {
        "date": "Sunday 2026-07-05",
        "uk_time": "21:00",
        "venue": "MetLife Stadium",
        "city": "East Rutherford",
        "stage": "Round of 16",
        "slot1": "Winner 76",
        "slot2": "Winner 78",
    },

    92: {
        "date": "Monday 2026-07-06",
        "uk_time": "01:00",
        "venue": "Estadio Azteca",
        "city": "Mexico City",
        "stage": "Round of 16",
        "slot1": "Winner 79",
        "slot2": "Winner 80",
    },

    93: {
        "date": "Monday 2026-07-06",
        "uk_time": "20:00",
        "venue": "AT&T Stadium",
        "city": "Arlington",
        "stage": "Round of 16",
        "slot1": "Winner 83",
        "slot2": "Winner 84",
    },

    94: {
        "date": "Tuesday 2026-07-07",
        "uk_time": "01:00",
        "venue": "Lumen Field",
        "city": "Seattle",
        "stage": "Round of 16",
        "slot1": "Winner 81",
        "slot2": "Winner 82",
    },

    95: {
        "date": "Tuesday 2026-07-07",
        "uk_time": "17:00",
        "venue": "Mercedes-Benz Stadium",
        "city": "Atlanta",
        "stage": "Round of 16",
        "slot1": "Winner 86",
        "slot2": "Winner 88",
    },

    96: {
        "date": "Tuesday 2026-07-07",
        "uk_time": "21:00",
        "venue": "BC Place",
        "city": "Vancouver",
        "stage": "Round of 16",
        "slot1": "Winner 85",
        "slot2": "Winner 87",
    },

    97: {
        "date": "Thursday 2026-07-09",
        "uk_time": "21:00",
        "venue": "Gillette Stadium",
        "city": "Foxborough",
        "stage": "Quarterfinals",
        "slot1": "Winner 89",
        "slot2": "Winner 90",
    },

    98: {
        "date": "Friday 2026-07-10",
        "uk_time": "20:00",
        "venue": "SoFi Stadium",
        "city": "Inglewood",
        "stage": "Quarterfinals",
        "slot1": "Winner 93",
        "slot2": "Winner 94",
    },

    99: {
        "date": "Saturday 2026-07-11",
        "uk_time": "22:00",
        "venue": "Hard Rock Stadium",
        "city": "Miami Gardens",
        "stage": "Quarterfinals",
        "slot1": "Winner 91",
        "slot2": "Winner 92",
    },

    100: {
        "date": "Sunday 2026-07-12",
        "uk_time": "02:00",
        "venue": "Arrowhead Stadium",
        "city": "Kansas City",
        "stage": "Quarterfinals",
        "slot1": "Winner 95",
        "slot2": "Winner 96",
    },

    101: {
        "date": "Tuesday 2026-07-14",
        "uk_time": "20:00",
        "venue": "AT&T Stadium",
        "city": "Arlington",
        "stage": "Semifinals",
        "slot1": "Winner 97",
        "slot2": "Winner 98",
    },

    102: {
        "date": "Wednesday 2026-07-15",
        "uk_time": "20:00",
        "venue": "Mercedes-Benz Stadium",
        "city": "Atlanta",
        "stage": "Semifinals",
        "slot1": "Winner 99",
        "slot2": "Winner 100",
    },
    
    103: {
        "date": "Saturday 2026-07-18",
        "uk_time": "22:00",
        "venue": "Hard Rock Stadium",
        "city": "Miami Gardens",
        "stage": "Third-place Playoff",
        "slot1": "Loser 101",
        "slot2": "Loser 102",
    },

    104: {
        "date": "Sunday 2026-07-19",
        "uk_time": "20:00",
        "venue": "MetLife Stadium",
        "city": "New York/New Jersey",
        "stage": "Final",
        "slot1": "Winner 101",
        "slot2": "Winner 102",
    },
}


# ============================================================
# VALIDATION
# ============================================================

def validate_teams():
    missing = []
    for group, teams in groups.items():
        for team in teams:
            if team not in base_team_elo:
                missing.append(team)

    if missing:
        raise ValueError(f"Missing Elo ratings for: {missing}")

validate_teams()

# ============================================================
# THIRD-PLACE FIFA MAPPING TABLE
# ============================================================

def load_third_place_mapping_table():
    import requests
    from io import StringIO

    api_url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "parse",
        "page": "Template:2026 FIFA World Cup third-place table",
        "prop": "text",
        "format": "json",
        "formatversion": 2,
    }

    headers = {
        "User-Agent": (
            "WorldCupSimulator/1.0 "
            "(https://github.com/GeorgelHuja/World-Cup-2026-Simulator)"
        )
    }

    response = requests.get(
        api_url,
        params=params,
        headers=headers,
        timeout=30,
    )

    response.raise_for_status()

    html = response.json()["parse"]["text"]
    tables = pd.read_html(StringIO(html))
    df = tables[0]

    mapping_table = {}

    for _, row in df.iterrows():
        cells = [
            str(x).strip()
            for x in row.tolist()
            if str(x).strip() not in ["nan", ""]
        ]

        groups_found = []
        assignments = []

        for cell in cells:
            if re.fullmatch(r"[A-L]", cell):
                groups_found.append(cell)

            elif re.fullmatch(r"3[A-L]", cell):
                assignments.append(cell)

        if len(groups_found) == 8 and len(assignments) == 8:
            key = tuple(sorted(groups_found))
            mapping_table[key] = dict(zip(THIRD_PLACE_SLOTS, assignments))

    if len(mapping_table) != 495:
        print(f"Loaded {len(mapping_table)} mappings instead of 495.")
        print("First few rows:")
        print(df.head())
        raise ValueError(f"Expected 495 third-place mappings, got {len(mapping_table)}")

    return mapping_table


THIRD_PLACE_MAPPING_TABLE = load_third_place_mapping_table()

# ============================================================
# POISSON GOAL MODEL
# ============================================================

def poisson_sample(lam):
    L = math.exp(-lam)
    k = 0
    p = 1.0

    while p > L:
        k += 1
        p *= random.random()

    return k - 1


def adjusted_elo(team, team_elo):
    return team_elo[team] + HOST_BONUS.get(team, 0)


def expected_goals(team_for, team_against, team_elo):
    elo_for = adjusted_elo(team_for, team_elo)
    elo_against = adjusted_elo(team_against, team_elo)

    elo_diff = elo_for - elo_against

    base_xg = 1.35
    elo_effect = elo_diff / 400

    xg = base_xg + elo_effect

    return max(0.15, min(xg, 3.5))


def simulate_score(team1, team2, team_elo):
    xg1 = expected_goals(team1, team2, team_elo)
    xg2 = expected_goals(team2, team1, team_elo)

    goals1 = poisson_sample(xg1)
    goals2 = poisson_sample(xg2)

    return goals1, goals2

# ============================================================
# ELO UPDATE
# ============================================================

def expected_result(team1, team2, team_elo):
    elo1 = adjusted_elo(team1, team_elo)
    elo2 = adjusted_elo(team2, team_elo)

    return 1 / (1 + 10 ** ((elo2 - elo1) / 400))


def update_elo(team1, team2, goals1, goals2, team_elo, k):
    rating1 = team_elo[team1]
    rating2 = team_elo[team2]

    exp1 = expected_result(team1, team2, team_elo)
    exp2 = 1 - exp1

    if goals1 > goals2:
        score1, score2 = 1, 0
    elif goals2 > goals1:
        score1, score2 = 0, 1
    else:
        score1, score2 = 0.5, 0.5

    margin = abs(goals1 - goals2)
    mov_multiplier = 1 + margin * 0.25

    change1 = k * mov_multiplier * (score1 - exp1)
    change2 = k * mov_multiplier * (score2 - exp2)

    team_elo[team1] = round(rating1 + change1)
    team_elo[team2] = round(rating2 + change2)


def play_match(team1, team2, team_elo, k, update_ratings=True, verbose=False):
    goals1, goals2 = simulate_score(team1, team2, team_elo)

    if verbose:
        print(f"{team1} {goals1} - {goals2} {team2}")

    if update_ratings:
        update_elo(team1, team2, goals1, goals2, team_elo, k)

    if goals1 > goals2:
        return goals1, goals2, 3, 0
    elif goals2 > goals1:
        return goals1, goals2, 0, 3
    else:
        return goals1, goals2, 1, 1

def play_scheduled_match(match_id, team1, team2, team_elo, round_size, verbose=False):
    match_info = MATCH_SCHEDULE[match_id]

    if verbose:
        print(
            f"\nMatch {match_id} | {match_info['date']} | "
            f"{match_info['uk_time']} UK | {match_info['venue']}, {match_info['city']}"
        )

    return knockout_match(team1, team2, team_elo, round_size, verbose)
# ============================================================
# PENALTIES
# ============================================================

def penalty_shootout(team1, team2, verbose=False):
    team1_score = 0
    team2_score = 0
    team1_kicks = 0
    team2_kicks = 0

    for round_num in range(5):
        kick1 = random.choices([1, 0], weights=[78, 22])[0]
        kick2 = random.choices([1, 0], weights=[78, 22])[0]

        team1_score += kick1
        team2_score += kick2
        team1_kicks += 1
        team2_kicks += 1

        remaining1 = 5 - team1_kicks
        remaining2 = 5 - team2_kicks

        if verbose:
            print(f"Penalty round {round_num + 1}: {team1_score} - {team2_score}")

        if team1_score > team2_score + remaining2:
            return team1
        if team2_score > team1_score + remaining1:
            return team2

    while team1_score == team2_score:
        kick1 = random.choices([1, 0], weights=[78, 22])[0]
        kick2 = random.choices([1, 0], weights=[78, 22])[0]

        team1_score += kick1
        team2_score += kick2

    return team1 if team1_score > team2_score else team2

# ============================================================
# GROUP STAGE
# ============================================================

def simulate_group_stage(team_elo, verbose=False):
    standings = {
        group: {
            team: {
                "points": 0,
                "goal_diff": 0,
                "goals_scored": 0,
                "goals_against": 0,
            }
            for team in teams
        }
        for group, teams in groups.items()
    }

    for group, teams in groups.items():
        if verbose:
            print(f"\nGroup {group}")

        for i in range(len(teams)):
            for j in range(i + 1, len(teams)):
                team1 = teams[i]
                team2 = teams[j]

                g1, g2, p1, p2 = play_match(
                    team1,
                    team2,
                    team_elo,
                    k=BASE_K_GROUP,
                    update_ratings=True,
                    verbose=verbose,
                )

                standings[group][team1]["points"] += p1
                standings[group][team2]["points"] += p2

                standings[group][team1]["goal_diff"] += g1 - g2
                standings[group][team2]["goal_diff"] += g2 - g1

                standings[group][team1]["goals_scored"] += g1
                standings[group][team2]["goals_scored"] += g2

                standings[group][team1]["goals_against"] += g2
                standings[group][team2]["goals_against"] += g1

    return standings


def sort_group(group_table):
    return sorted(
        group_table.items(),
        key=lambda x: (
            -x[1]["points"],
            -x[1]["goal_diff"],
            -x[1]["goals_scored"],
            x[1]["goals_against"],
            random.random(),
        ),
    )


def get_qualified_slots(standings):
    qualified = {}
    third_place_teams = []

    for group, table in standings.items():
        sorted_teams = sort_group(table)

        qualified[f"1{group}"] = sorted_teams[0][0]
        qualified[f"2{group}"] = sorted_teams[1][0]
        qualified[f"3{group}"] = sorted_teams[2][0]

        third_team, third_stats = sorted_teams[2]
        third_place_teams.append((group, third_team, third_stats))

    third_place_teams.sort(
        key=lambda x: (
            -x[2]["points"],
            -x[2]["goal_diff"],
            -x[2]["goals_scored"],
            x[2]["goals_against"],
            random.random(),
        )
    )

    best_third_groups = sorted([group for group, team, stats in third_place_teams[:8]])

    return qualified, best_third_groups

# ============================================================
# MONTE CARLO SIMS
# ============================================================
def matchup_key(team1, team2):
    return tuple(sorted([team1, team2]))
# ============================================================
# OFFICIAL 2026 KNOCKOUT BRACKET
# ============================================================

def build_official_round_of_32(qualified, best_third_groups):
    key = tuple(best_third_groups)

    if key not in THIRD_PLACE_MAPPING_TABLE:
        raise ValueError(f"No third-place mapping found for groups: {key}")

    third_mapping = THIRD_PLACE_MAPPING_TABLE[key]

    return {
        73: ("2A", "2B"),
        74: ("1E", third_mapping["1E"]),
        75: ("1F", "2C"),
        76: ("1C", "2F"),
        77: ("1I", third_mapping["1I"]),
        78: ("2E", "2I"),
        79: ("1A", third_mapping["1A"]),
        80: ("1L", third_mapping["1L"]),
        81: ("1G", third_mapping["1G"]),
        82: ("1D", third_mapping["1D"]),
        83: ("2K", "2L"),
        84: ("1H", "2J"),
        85: ("1B", third_mapping["1B"]),
        86: ("1J", "2H"),
        87: ("1K", third_mapping["1K"]),
        88: ("2D", "2G"),
    }


def knockout_match(team1, team2, team_elo, round_size, verbose=False):
    if round_size == 2:
        k = BASE_K_FINAL
    else:
        k = BASE_K_KNOCKOUT

    g1, g2, _, _ = play_match(
        team1,
        team2,
        team_elo,
        k=k,
        update_ratings=True,
        verbose=verbose,
    )

    if g1 > g2:
        return team1
    elif g2 > g1:
        return team2
    else:
        winner = penalty_shootout(team1, team2, verbose=verbose)

        if verbose:
            print(f"{team1} {g1}-{g2} {team2}: {winner} wins on penalties")

        return winner


def simulate_official_knockouts(qualified, best_third_groups, team_elo, verbose=False, matchup_tracker=None):
    round_of_32 = build_official_round_of_32(qualified, best_third_groups)

    winners = {}
    
    if verbose:
        print("\nROUND OF 32")

    for match_id in range(73, 89):
        slot1, slot2 = round_of_32[match_id]
        team1 = qualified[slot1]
        team2 = qualified[slot2]

        if matchup_tracker is not None:
            matchup_tracker["Round of 32"][matchup_key(team1, team2)] += 1

        winner = play_scheduled_match(
            match_id,
            team1,
            team2,
            team_elo,
            round_size=32,
            verbose=verbose,
        )

        winners[match_id] = winner

        if verbose:
            print(f"Match {match_id}: {team1} vs {team2} -> {winner}")

        # Round of 16 through Semifinals
    bracket_paths = {
        89: (74, 77),
        90: (73, 75),
        91: (76, 78),
        92: (79, 80),
        93: (83, 84),
        94: (81, 82),
        95: (86, 88),
        96: (85, 87),

        97: (89, 90),
        98: (93, 94),
        99: (91, 92),
        100: (95, 96),

        101: (97, 98),
        102: (99, 100),
    }

    round_sizes = {
        89: 16, 90: 16, 91: 16, 92: 16,
        93: 16, 94: 16, 95: 16, 96: 16,
        97: 8, 98: 8, 99: 8, 100: 8,
        101: 4, 102: 4,
    }

    round_names = {
        89: "ROUND OF 16",
        97: "QUARTERFINALS",
        101: "SEMIFINALS",
    }

    losers = {}
    
    for match_id, previous_matches in bracket_paths.items():
        if verbose and match_id in round_names:
            print(f"\n{round_names[match_id]}")

        team1 = winners[previous_matches[0]]
        team2 = winners[previous_matches[1]]

        round_label = {
            16: "Round of 16",
            8: "Quarterfinals",
            4: "Semifinals",
        }[round_sizes[match_id]]

        if matchup_tracker is not None:
            matchup_tracker[round_label][matchup_key(team1, team2)] += 1

        winner = play_scheduled_match(
            match_id,
            team1,
            team2,
            team_elo,
            round_size=round_sizes[match_id],
            verbose=verbose,
        )

        loser = team2 if winner == team1 else team1

        winners[match_id] = winner
        losers[match_id] = loser

        if verbose:
            print(f"Match {match_id}: {team1} vs {team2} -> {winner}")

    # Match 103: Third-place playoff
    if verbose:
        print("\nTHIRD-PLACE PLAYOFF")

    if matchup_tracker is not None:
        matchup_tracker["Third-place Playoff"][matchup_key(losers[101], losers[102])] += 1

    third_place_winner = knockout_match(
        losers[101],
        losers[102],
        team_elo,
        round_size=2,
        verbose=verbose,
    )

    winners[103] = third_place_winner

    if verbose:
        print(f"Match 103: {losers[101]} vs {losers[102]} -> {third_place_winner}")

    # Match 104: Final
    if verbose:
        print("\nFINAL")

    if matchup_tracker is not None:
        matchup_tracker["Final"][matchup_key(winners[101], winners[102])] += 1

    champion = knockout_match(
        winners[101],
        winners[102],
        team_elo,
        round_size=2,
        verbose=verbose,
    )

    winners[104] = champion

    if verbose:
        print(f"Match 104: {winners[101]} vs {winners[102]} -> {champion}")

    return winners[104]
# ============================================================
# FULL TOURNAMENT
# ============================================================

def simulate_tournament(verbose=False, matchup_tracker=None):
    team_elo = deepcopy(base_team_elo)

    standings = simulate_group_stage(team_elo, verbose=verbose)

    if verbose:
        print("\nFINAL GROUP STANDINGS")

        for group, table in standings.items():
            print(f"\nGroup {group}")
            print("Team                 Pts  GD  GF  GA")

            for team, stats in sort_group(table):
                print(
                    f"{team:<20} "
                    f"{stats['points']:>2}   "
                    f"{stats['goal_diff']:>2}   "
                    f"{stats['goals_scored']:>2}   "
                    f"{stats['goals_against']:>2}"
                )

    qualified, best_third_groups = get_qualified_slots(standings)

    if verbose:
        print("\nBest third-place groups:")
        print(best_third_groups)

        print("\nQualified slots:")
        for slot in sorted(qualified.keys()):
            if slot.startswith("3") and slot[1] not in best_third_groups:
                continue
            print(f"{slot}: {qualified[slot]}")

    champion = simulate_official_knockouts(
        qualified,
        best_third_groups,
        team_elo,
        verbose=verbose,
        matchup_tracker=matchup_tracker,
    )

    if verbose:
        print(f"\nWORLD CUP CHAMPION: {champion}")

    return champion

# ============================================================
# MONTE CARLO
# ============================================================

def monte_carlo_simulation(num_simulations=10000):
    champions = defaultdict(int)

    matchup_tracker = {
        "Round of 32": defaultdict(int),
        "Round of 16": defaultdict(int),
        "Quarterfinals": defaultdict(int),
        "Semifinals": defaultdict(int),
        "Third-place Playoff": defaultdict(int),
        "Final": defaultdict(int),
    }

    for _ in range(num_simulations):
        champion = simulate_tournament(
            verbose=False,
            matchup_tracker=matchup_tracker
        )
        champions[champion] += 1

    print(f"\nMONTE CARLO RESULTS: {num_simulations:,} simulations\n")
    print("Team                 Titles   Chance")

    for team, wins in sorted(champions.items(), key=lambda x: x[1], reverse=True):
        chance = wins / num_simulations * 100
        print(f"{team:<20} {wins:>6}   {chance:>6.2f}%")

    print("\nMOST LIKELY KNOCKOUT MATCHUPS\n")

    for round_name, matchups in matchup_tracker.items():
        print(round_name)
        print("Matchup                              Times   Chance")

        top_matchups = sorted(
            matchups.items(),
            key=lambda x: x[1],
            reverse=True
        )[:8]

        for (team1, team2), count in top_matchups:
            chance = count / num_simulations * 100
            matchup = f"{team1} vs {team2}"
            print(f"{matchup:<36} {count:>5}   {chance:>6.2f}%")

        print()

# ============================================================
# RUN
# ============================================================

if PRINT_SINGLE_TOURNAMENT:
    simulate_tournament(verbose=True)

monte_carlo_simulation(NUM_SIMULATIONS)


Group A
Mexico 3 - 0 South Africa
Mexico 1 - 0 South Korea
Mexico 3 - 2 Czech Republic
South Africa 1 - 2 South Korea
South Africa 0 - 2 Czech Republic
South Korea 1 - 0 Czech Republic

Group B
Canada 0 - 0 Bosnia and Herzegovina
Canada 2 - 1 Qatar
Canada 1 - 2 Switzerland
Bosnia and Herzegovina 0 - 1 Qatar
Bosnia and Herzegovina 0 - 1 Switzerland
Qatar 0 - 4 Switzerland

Group C
Brazil 0 - 0 Morocco
Brazil 5 - 0 Haiti
Brazil 2 - 1 Scotland
Morocco 1 - 0 Haiti
Morocco 4 - 1 Scotland
Haiti 0 - 0 Scotland

Group D
USA 5 - 4 Paraguay
USA 0 - 1 Australia
USA 1 - 2 Turkiye
Paraguay 2 - 0 Australia
Paraguay 0 - 2 Turkiye
Australia 1 - 0 Turkiye

Group E
Germany 0 - 0 Curacao
Germany 2 - 1 Ivory Coast
Germany 1 - 0 Ecuador
Curacao 1 - 2 Ivory Coast
Curacao 0 - 4 Ecuador
Ivory Coast 0 - 1 Ecuador

Group F
Netherlands 1 - 0 Japan
Netherlands 2 - 1 Sweden
Netherlands 1 - 0 Tunisia
Japan 2 - 0 Sweden
Japan 3 - 0 Tunisia
Sweden 1 - 0 Tunisia

Group G
Belgium 3 - 0 Egypt
Belgium 3 - 2 Iran
Belgium